In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import timm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# UPDATED PATHS for new structure
DRIVE_BASE = "/content/drive/MyDrive/ThangNQ-NamBH/fitzpatrick_dataset"
TRAIN_CSV = os.path.join(DRIVE_BASE, "metadata.csv")
IMG_DIR = os.path.join(DRIVE_BASE, "images")

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 1e-4
IMG_SIZE = 256

# Split ratios
TRAIN_RATIO = 0.75
VAL_RATIO = 0.15
TEST_RATIO = 0.1

print(f"Configuration:")
print(f"- Batch Size: {BATCH_SIZE}")
print(f"- Epochs: {NUM_EPOCHS}")
print(f"- Learning Rate: {LEARNING_RATE}")
print(f"- Image Size: {IMG_SIZE}x{IMG_SIZE}")

In [ ]:
print("\n" + "="*50)
print("DATA PREPROCESSING")
print("="*50)

# Load and clean data
df = pd.read_csv(TRAIN_CSV)
print(f"Original dataset size: {len(df)}")

# Remove low quality images (qc=3)
if 'qc' in df.columns:
    df = df[df['qc'].astype(str).str.strip() != '3']
    print(f"After removing qc=3: {len(df)}")

# Check and remove missing images
print("\nChecking for missing images...")
valid_indices = []
missing_count = 0

for idx, row in df.iterrows():
    img_name = row['md5hash'] + ".jpg"
    img_path = os.path.join(IMG_DIR, img_name)

    if os.path.exists(img_path):
        valid_indices.append(idx)
    else:
        missing_count += 1

df = df.loc[valid_indices].reset_index(drop=True)
print(f"\n✓ Removed {missing_count} missing images")
print(f"✓ Final dataset size: {len(df)}")

# Check label distribution
print("\nLabel distribution:")
label_counts = df['label'].value_counts()
print(label_counts)

# Analyze class imbalance
print("\nClass Imbalance Analysis:")
max_samples = label_counts.max()
min_samples = label_counts.min()
imbalance_ratio = max_samples / min_samples
print(f"  Max class: {max_samples} samples")
print(f"  Min class: {min_samples} samples")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}x")

if imbalance_ratio > 10:
    print("HIGH IMBALANCE detected - consider using more augmentation or focal loss")

# Create label mapping
unique_labels = sorted(df['label'].unique())
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
num_classes = len(unique_labels)

print(f"\nNumber of classes: {num_classes}")
print(f"Classes: {unique_labels}")

# Encode labels
df['label_encoded'] = df['label'].map(label_to_idx)

# Split dataset: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(
    df, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['label_encoded'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=TEST_RATIO/(VAL_RATIO + TEST_RATIO),
    stratify=temp_df['label_encoded'],
    random_state=42
)
print(f"\n Deleted {missing_count}:")
print(f"\nDataset split:")
print(f"- Train: {len(train_df)} samples")
print(f"- Validation: {len(val_df)} samples")
print(f"- Test: {len(test_df)} samples")

In [ ]:
print("\n" + "="*50)
print("DATA AUGMENTATION SETUP")
print("="*50)

# Training transforms with augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test transforms without augmentation
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Augmentation techniques applied:")
print("- Horizontal & Vertical Flip")
print("- Random Rotation (±20°)")
print("- Color Jitter (brightness, contrast, saturation, hue)")
print("- Random Affine transformation")


DATA AUGMENTATION SETUP
Augmentation techniques applied:
- Horizontal & Vertical Flip
- Random Rotation (±20°)
- Color Jitter (brightness, contrast, saturation, hue)
- Random Affine transformation


In [ ]:
class SkinDiseaseDataset(Dataset):
    """UPDATED: Images are now in a single folder"""
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['md5hash'] + ".jpg"
        # UPDATED: Direct path to image (no label subfolder)
        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image if error
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row['label_encoded'], dtype=torch.long)
        return image, label

# Create datasets
train_dataset = SkinDiseaseDataset(train_df, IMG_DIR, transform=train_transform)
val_dataset = SkinDiseaseDataset(val_df, IMG_DIR, transform=val_transform)
test_dataset = SkinDiseaseDataset(test_df, IMG_DIR, transform=val_transform)

print(f"\nDatasets created successfully")


Datasets created successfully


In [ ]:
print("\n" + "="*50)
print("HANDLING CLASS IMBALANCE")
print("="*50)

# Compute class weights
labels = train_df['label_encoded'].values
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:")
for label, weight in zip(unique_labels, class_weights):
    print(f"  {label}: {weight:.4f}")

# Create weighted sampler
sample_weights = np.array([class_weights[label] for label in labels])
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"\nData loaders created with batch size: {BATCH_SIZE}")


In [ ]:
print("\n" + "="*50)
print("BUILDING FUSION MODEL")
print("="*50)

class SkinDiseaseFusionModel(nn.Module):
    """
    Fusion model combining EfficientNet-B0, EfficientNet-B2, and ResNet50
    """
    def __init__(self, num_classes):
        super(SkinDiseaseFusionModel, self).__init__()

        # Load pre-trained models
        self.efficientnet_b0 = timm.create_model('efficientnet_b0', pretrained=True)
        self.efficientnet_b2 = timm.create_model('efficientnet_b2', pretrained=True)
        self.resnet50 = timm.create_model('resnet50', pretrained=True)

        # Get feature dimensions
        self.b0_features = self.efficientnet_b0.num_features
        self.b2_features = self.efficientnet_b2.num_features
        self.resnet_features = self.resnet50.num_features

        # Remove classification heads
        self.efficientnet_b0.classifier = nn.Identity()
        self.efficientnet_b2.classifier = nn.Identity()
        self.resnet50.fc = nn.Identity()

        # Total fused features
        self.total_features = self.b0_features + self.b2_features + self.resnet_features

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.total_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Extract features from each model
        feat_b0 = self.efficientnet_b0(x)
        feat_b2 = self.efficientnet_b2(x)
        feat_resnet = self.resnet50(x)

        # Concatenate features (fusion)
        fused_features = torch.cat([feat_b0, feat_b2, feat_resnet], dim=1)

        # Classification
        output = self.classifier(fused_features)

        return output

# Initialize model
model = SkinDiseaseFusionModel(num_classes=num_classes)
model = model.to(device)

print("Fusion Model Architecture:")
print(f"- EfficientNet-B0 features: {model.b0_features}")
print(f"- EfficientNet-B2 features: {model.b2_features}")
print(f"- ResNet50 features: {model.resnet_features}")
print(f"- Total fused features: {model.total_features}")
print(f"- Classification layers: 512 -> 256 -> {num_classes}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


BUILDING FUSION MODEL


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Fusion Model Architecture:
- EfficientNet-B0 features: 1280
- EfficientNet-B2 features: 1408
- ResNet50 features: 2048
- Total fused features: 4736
- Classification layers: 512 -> 256 -> 116

Total parameters: 37,803,058
Trainable parameters: 37,803,058


In [ ]:
print("\n" + "="*50)
print("TRAINING SETUP")
print("="*50)

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer: AdamW
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print(f"Loss: CrossEntropyLoss with class weights")
print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc



TRAINING SETUP
Loss: CrossEntropyLoss with class weights
Optimizer: AdamW (lr=0.0001, weight_decay=0.0001)
Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)


In [ ]:
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

best_val_acc = 0.0
best_model_path = 'best_fusion_model.pth'

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 50)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)

    # Update scheduler
    scheduler.step(val_loss)

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Print results
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"✓ Best model saved with val_acc: {val_acc:.2f}%")

print("\n" + "="*50)
print("TRAINING COMPLETED")
print("="*50)
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")


In [ ]:
print("\n" + "="*50)
print("TRAINING VISUALIZATION")
print("="*50)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history plots saved as 'training_history.png'")

In [ ]:
print("\n" + "="*50)
print("EVALUATION ON TEST SET")
print("="*50)

# Load best model
model.load_state_dict(torch.load(best_model_path))
model.eval()

# Collect predictions
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Calculate metrics
test_accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted', zero_division=0
)

print(f"\nTest Set Performance:")
print(f"- Accuracy: {test_accuracy*100:.2f}%")
print(f"- Precision: {precision:.4f}")
print(f"- Recall: {recall:.4f}")
print(f"- F1-Score: {f1:.4f}")

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=unique_labels,
    zero_division=0
))

In [ ]:
print("\n" + "="*50)
print("CONFUSION MATRIX")
print("="*50)

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=unique_labels,
    yticklabels=unique_labels
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved as 'confusion_matrix.png'")

In [ ]:
print("\n" + "="*50)
print("ROC-AUC ANALYSIS")
print("="*50)

# Binarize labels for ROC-AUC
y_test_bin = label_binarize(all_labels, classes=range(num_classes))

# Calculate AUC for each class
auc_scores = []
for i in range(num_classes):
    try:
        auc = roc_auc_score(y_test_bin[:, i], all_probs[:, i])
        auc_scores.append(auc)
        print(f"{unique_labels[i]}: AUC = {auc:.4f}")
    except:
        auc_scores.append(0.0)
        print(f"{unique_labels[i]}: AUC = N/A")

mean_auc = np.mean([s for s in auc_scores if s > 0])
print(f"\nMean AUC-ROC: {mean_auc:.4f}")


In [ ]:
print("\n" + "="*50)
print("SAVING MODEL & METADATA")
print("="*50)

# Save final model
final_model_path = 'skin_disease_fusion_model_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'label_mapping': label_to_idx,
    'idx_to_label': idx_to_label,
    'num_classes': num_classes,
    'best_val_acc': best_val_acc,
    'test_accuracy': test_accuracy,
    'history': history
}, final_model_path)

print(f"✓ Model saved to: {final_model_path}")
print(f"✓ Best validation accuracy: {best_val_acc:.2f}%")
print(f"✓ Test accuracy: {test_accuracy*100:.2f}%")

# Save to Drive (optional)
import shutil
drive_save_path = os.path.join(DRIVE_BASE, "models")
os.makedirs(drive_save_path, exist_ok=True)
shutil.copy(final_model_path, os.path.join(drive_save_path, final_model_path))
shutil.copy('training_history.png', os.path.join(drive_save_path, 'training_history.png'))
shutil.copy('confusion_matrix.png', os.path.join(drive_save_path, 'confusion_matrix.png'))

print(f"\n✓ All files backed up to Google Drive: {drive_save_path}")

print("\n" + "="*50)
print("TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
print("="*50)